# Module 2 – Data Preprocessing

## Bootcamp Final Project

### Objective
The objective of this module is to clean, standardize, and prepare the unified football dataset produced in Module 1 for analysis and modeling. This includes handling missing values, fixing data types, selecting relevant features, and ensuring data consistency.

### Input Data
- Unified raw dataset generated in Module 1
- Source: football match data across multiple seasons

### Output
- Cleaned and preprocessed dataset
- Saved to: data/interim/ or data/silver/


In [1]:
from pathlib import Path

# Start from current working directory
here = Path.cwd()

# Sometimes VS Code/Jupyter sets cwd strangely; make it a directory for sure
if here.suffix == ".ipynb":
    here = here.parent

# Go up until we find the folder that contains "data"
project_root = here
while not (project_root / "data").exists() and project_root != project_root.parent:
    project_root = project_root.parent

print("HERE:", here)
print("PROJECT_ROOT:", project_root)
print("Has data folder?", (project_root / "data").exists())


HERE: c:\Users\Public\Bootcamp_FinalProject\project_repo\notebooks
PROJECT_ROOT: c:\Users\Public\Bootcamp_FinalProject\project_repo\notebooks
Has data folder? True


In [5]:
from pathlib import Path

project_root = Path(r"C:\Users\Public\Bootcamp_FinalProject\project_repo")

RAW_ROOT = project_root / "data" / "raw" / "football_data" / "mmz4281"
SILVER_DIR = project_root / "data" / "silver"

print("RAW exists?", RAW_ROOT.exists())
print("SILVER exists?", SILVER_DIR.exists())


RAW exists? True
SILVER exists? True


In [6]:
import pandas as pd

dfs = []

for season_folder in RAW_ROOT.iterdir():
    if season_folder.is_dir():
        csv_files = list(season_folder.glob("*.csv"))
        if csv_files:
            df_season = pd.read_csv(csv_files[0])
            df_season["season"] = season_folder.name
            dfs.append(df_season)

df = pd.concat(dfs, ignore_index=True)

print("df shape:", df.shape)
df.head()


df shape: (3800, 153)


,Div,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,HTR,...,1XBCH,1XBCD,1XBCA,BFECH,BFECD,BFECA,BFEC>2.5,BFEC<2.5,BFECAHH,BFECAHA
0,SP1,21/08/15,Malaga,Sevilla,0,0,D,0,0,D,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,SP1,22/08/15,Ath Madrid,Las Palmas,1,0,H,1,0,H,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,SP1,22/08/15,Espanol,Getafe,1,0,H,1,0,H,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,SP1,22/08/15,La Coruna,Sociedad,0,0,D,0,0,D,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,SP1,22/08/15,Vallecano,Valencia,0,0,D,0,0,D,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
df_raw = df.copy()
df = df_raw.copy()
print(df.shape)


(3800, 153)


In [8]:
core_cols = [
    "season", "Div", "Date", "HomeTeam", "AwayTeam",
    "FTHG", "FTAG", "FTR", "HTHG", "HTAG", "HTR"
]

# we only keep the columns that they really exist
core_cols = [c for c in core_cols if c in df.columns]

df_core = df[core_cols].copy()
print(df_core.shape)
df_core.head()


(3800, 11)


,season,Div,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,HTR
0,1516,SP1,21/08/15,Malaga,Sevilla,0,0,D,0,0,D
1,1516,SP1,22/08/15,Ath Madrid,Las Palmas,1,0,H,1,0,H
2,1516,SP1,22/08/15,Espanol,Getafe,1,0,H,1,0,H
3,1516,SP1,22/08/15,La Coruna,Sociedad,0,0,D,0,0,D
4,1516,SP1,22/08/15,Vallecano,Valencia,0,0,D,0,0,D


In [9]:
df_core["Date"] = pd.to_datetime(df_core["Date"], dayfirst=True, errors="coerce")
df_core["Date"].isna().sum()


C:\Users\mfsht\AppData\Local\Temp\ipykernel_33612\1934072345.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_core["Date"] = pd.to_datetime(df_core["Date"], dayfirst=True, errors="coerce")


np.int64(0)

In [10]:
for col in ["FTHG", "FTAG", "HTHG", "HTAG"]:
    if col in df_core.columns:
        df_core[col] = pd.to_numeric(df_core[col], errors="coerce")


In [11]:
required = ["season", "Date", "HomeTeam", "AwayTeam", "FTHG", "FTAG", "FTR"]
required = [c for c in required if c in df_core.columns]

before = df_core.shape[0]
df_core = df_core.dropna(subset=required)
after = df_core.shape[0]

print("Rows before:", before)
print("Rows after :", after)
print("Dropped   :", before - after)


Rows before: 3800
Rows after : 3800
Dropped   : 0


In [12]:
before = df_core.shape[0]
df_core = df_core.drop_duplicates()
after = df_core.shape[0]
print("Duplicates removed:", before - after)


Duplicates removed: 0


In [13]:
df_core["goal_diff"] = df_core["FTHG"] - df_core["FTAG"]
df_core.head()


,season,Div,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,HTR,goal_diff
0,1516,SP1,2015-08-21,Malaga,Sevilla,0,0,D,0,0,D,0
1,1516,SP1,2015-08-22,Ath Madrid,Las Palmas,1,0,H,1,0,H,1
2,1516,SP1,2015-08-22,Espanol,Getafe,1,0,H,1,0,H,1
3,1516,SP1,2015-08-22,La Coruna,Sociedad,0,0,D,0,0,D,0
4,1516,SP1,2015-08-22,Vallecano,Valencia,0,0,D,0,0,D,0


In [14]:
output_path = Path("../data/interim")
output_path.mkdir(parents=True, exist_ok=True)

file_out = output_path / "matches_core_clean.parquet"
df_core.to_parquet(file_out, index=False)

print("Saved to:", file_out)


Saved to: ..\data\interim\matches_core_clean.parquet


In [15]:
df_core.to_csv(output_path / "matches_core_clean.csv", index=False)


In [16]:
from pathlib import Path
import pandas as pd

# We are inside: project_repo/notebooks
# So project_repo is one level up
PROJECT_ROOT = Path.cwd().parents[0]  # this is .../project_repo/notebooks
PROJECT_REPO = Path.cwd().parents[1]  # this is .../project_repo

DATA_DIR = PROJECT_REPO / "data"
RAW_DIR = DATA_DIR / "raw" / "football_data" / "mmz4281"
INTERIM_DIR = DATA_DIR / "interim"
SILVER_DIR = DATA_DIR / "silver"

print("PROJECT_REPO:", PROJECT_REPO)
print("DATA_DIR exists:", DATA_DIR.exists())
print("RAW_DIR exists:", RAW_DIR.exists())
print("INTERIM_DIR exists:", INTERIM_DIR.exists())
print("SILVER_DIR exists:", SILVER_DIR.exists())


PROJECT_REPO: c:\Users\Public\Bootcamp_FinalProject\project_repo
DATA_DIR exists: True
RAW_DIR exists: True
INTERIM_DIR exists: True
SILVER_DIR exists: True


In [17]:
dfs = []

for season_folder in RAW_DIR.iterdir():
    if season_folder.is_dir():
        csv_files = list(season_folder.glob("*.csv"))
        if csv_files:
            df_season = pd.read_csv(csv_files[0])
            df_season["season"] = season_folder.name
            dfs.append(df_season)

df_raw = pd.concat(dfs, ignore_index=True)

print("df_raw shape:", df_raw.shape)
df_raw.head()


df_raw shape: (3800, 153)


,Div,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,HTR,...,1XBCH,1XBCD,1XBCA,BFECH,BFECD,BFECA,BFEC>2.5,BFEC<2.5,BFECAHH,BFECAHA
0,SP1,21/08/15,Malaga,Sevilla,0,0,D,0,0,D,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,SP1,22/08/15,Ath Madrid,Las Palmas,1,0,H,1,0,H,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,SP1,22/08/15,Espanol,Getafe,1,0,H,1,0,H,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,SP1,22/08/15,La Coruna,Sociedad,0,0,D,0,0,D,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,SP1,22/08/15,Vallecano,Valencia,0,0,D,0,0,D,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [18]:
df = df_raw.copy()

core_cols = ["season", "Div", "Date", "HomeTeam", "AwayTeam", "FTHG", "FTAG", "FTR", "HTHG", "HTAG", "HTR"]
core_cols = [c for c in core_cols if c in df.columns]

df_core = df[core_cols].copy()
print("df_core shape:", df_core.shape)
df_core.head()


df_core shape: (3800, 11)


,season,Div,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,HTR
0,1516,SP1,21/08/15,Malaga,Sevilla,0,0,D,0,0,D
1,1516,SP1,22/08/15,Ath Madrid,Las Palmas,1,0,H,1,0,H
2,1516,SP1,22/08/15,Espanol,Getafe,1,0,H,1,0,H
3,1516,SP1,22/08/15,La Coruna,Sociedad,0,0,D,0,0,D
4,1516,SP1,22/08/15,Vallecano,Valencia,0,0,D,0,0,D


In [19]:
df_core["Date"] = pd.to_datetime(df_core["Date"], dayfirst=True, errors="coerce")

for col in ["FTHG", "FTAG", "HTHG", "HTAG"]:
    if col in df_core.columns:
        df_core[col] = pd.to_numeric(df_core[col], errors="coerce")

print("Invalid dates:", df_core["Date"].isna().sum())


Invalid dates: 0


C:\Users\mfsht\AppData\Local\Temp\ipykernel_33612\70985126.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_core["Date"] = pd.to_datetime(df_core["Date"], dayfirst=True, errors="coerce")


In [20]:
required = ["season", "Date", "HomeTeam", "AwayTeam", "FTHG", "FTAG", "FTR"]
required = [c for c in required if c in df_core.columns]

before = df_core.shape[0]
df_core = df_core.dropna(subset=required)
after = df_core.shape[0]

df_core = df_core.drop_duplicates()

df_core["goal_diff"] = df_core["FTHG"] - df_core["FTAG"]

print("Rows before:", before)
print("Rows after:", after)
print("Final shape:", df_core.shape)
df_core.head()


Rows before: 3800
Rows after: 3800
Final shape: (3800, 12)


,season,Div,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,HTR,goal_diff
0,1516,SP1,2015-08-21,Malaga,Sevilla,0,0,D,0,0,D,0
1,1516,SP1,2015-08-22,Ath Madrid,Las Palmas,1,0,H,1,0,H,1
2,1516,SP1,2015-08-22,Espanol,Getafe,1,0,H,1,0,H,1
3,1516,SP1,2015-08-22,La Coruna,Sociedad,0,0,D,0,0,D,0
4,1516,SP1,2015-08-22,Vallecano,Valencia,0,0,D,0,0,D,0


In [21]:
INTERIM_DIR.mkdir(parents=True, exist_ok=True)
SILVER_DIR.mkdir(parents=True, exist_ok=True)

file_interim = INTERIM_DIR / "matches_core_clean.parquet"
df_core.to_parquet(file_interim, index=False)
print("Saved interim:", file_interim)

file_silver = SILVER_DIR / "matches_core_clean.parquet"
df_core.to_parquet(file_silver, index=False)
print("Saved silver:", file_silver)


Saved interim: c:\Users\Public\Bootcamp_FinalProject\project_repo\data\interim\matches_core_clean.parquet
Saved silver: c:\Users\Public\Bootcamp_FinalProject\project_repo\data\silver\matches_core_clean.parquet
